# Loan Default Prediction

Predicts whether a loan will be paid off (`PAIDOFF`) or sent to collections
(`COLLECTION`), comparing three classifiers -- KNN, Logistic Regression, and
a Decision Tree -- then simulates a retraining pipeline that periodically
refits each model on newly augmented data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                           recall_score, f1_score, confusion_matrix,
                           classification_report, roc_auc_score)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

from imblearn.over_sampling import SMOTE

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

## Load and explore the data

Dataset: IBM Cognitive Class loan repayment lab data. If this URL is dead
by the time you're reading this (it's an old CDN link), swap in a local
copy of `loan_train.csv`.

In [ ]:
url = 'https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/ML0101ENv3/labs/loan_train.csv'
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("\nTarget variable distribution:")
print(df['loan_status'].value_counts())
print(f"PAIDOFF: {df['loan_status'].value_counts(normalize=True)['PAIDOFF']:.1%}")
print(f"COLLECTION: {df['loan_status'].value_counts(normalize=True)['COLLECTION']:.1%}")
print("\nFirst 3 samples:")
print(df.head(3))
print("\nMissing values:")
print(df.isnull().sum())

## Preprocessing

Fixes a data-entry typo in `education`, engineers a couple of date/gender
features, and one-hot encodes education (dropping one level to avoid the
dummy-variable trap).

In [ ]:
df_processed = df.copy()
df_processed['education'] = df_processed['education'].str.replace('Bechalor', 'Bachelor')
df_processed['effective_date'] = pd.to_datetime(df_processed['effective_date'])
df_processed['dayofweek'] = df_processed['effective_date'].dt.dayofweek
df_processed['weekend'] = (df_processed['dayofweek'] >= 5).astype(int)
df_processed['Gender'] = df_processed['Gender'].map({'male': 0, 'female': 1})

education_dummies = pd.get_dummies(df_processed['education'], prefix='edu')
education_dummies = education_dummies.iloc[:, :-1]

features_list = ['Principal', 'terms', 'age', 'Gender', 'weekend']
X = pd.concat([df_processed[features_list], education_dummies], axis=1)
y = df_processed['loan_status']

print(f"\nPreprocessing complete!")
print(f"Features: {list(X.columns)}")
print(f"Final shape: {X.shape}")

## Train/test split and feature scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nData Split:")
print(f"Training set: {X_train.shape} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Testing set:  {X_test.shape} ({len(X_test)/len(X)*100:.0f}%)")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, 'loan_scaler.pkl')

print(f"Feature scaling applied. Scaler saved as 'loan_scaler.pkl'")

## Balance classes with SMOTE

The training set is skewed toward `PAIDOFF`, which would bias every model
toward predicting the majority class. SMOTE oversamples the minority
class synthetically rather than just duplicating rows.

In [ ]:
print(f"\nClass distribution BEFORE SMOTE:")
print(pd.Series(y_train).value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"\nClass distribution AFTER SMOTE:")
print(pd.Series(y_train_balanced).value_counts().to_dict())

## Train and evaluate the three models

In [ ]:
models = {}
performance_results = []

def train_evaluate_save(model, model_name, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, pos_label='PAIDOFF'),
        'Recall': recall_score(y_test, y_pred, pos_label='PAIDOFF'),
        'F1_Score': f1_score(y_test, y_pred, average='weighted'),
        'ROC_AUC': roc_auc_score(y_test, y_prob) if y_prob is not None else 'N/A'
    }

    models[model_name] = model
    filename = f"{model_name.lower().replace(' ', '_')}_model.pkl"
    joblib.dump(model, filename)
    print(f"Model saved as: {filename}")

    print(f"\nClassification Report for {model_name}:")
    print(classification_report(y_test, y_pred))

    return metrics

In [ ]:
# Pick k via cross-validation on the TRAINING set only -- using X_test here
# would leak test-set information into model selection and inflate the
# accuracy we later report on that same test set.
k_values = [3, 5, 7, 9, 11]
knn_cv_scores = []
for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    cv_acc = cross_val_score(knn_temp, X_train_balanced, y_train_balanced,
                              cv=5, scoring='accuracy')
    knn_cv_scores.append(cv_acc.mean())

best_k = k_values[np.argmax(knn_cv_scores)]
print(f"Optimal K for KNN (5-fold CV on train set): {best_k} "
      f"(CV Accuracy: {max(knn_cv_scores):.3f})")

knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_metrics = train_evaluate_save(knn_model, "KNN",
                                  X_train_balanced, y_train_balanced,
                                  X_test_scaled, y_test)
performance_results.append(knn_metrics)

In [ ]:
lr_model = LogisticRegression(
    C=0.01,
    solver='liblinear',
    max_iter=1000,
    random_state=42
)
lr_metrics = train_evaluate_save(lr_model, "Logistic Regression",
                                X_train_balanced, y_train_balanced,
                                X_test_scaled, y_test)
performance_results.append(lr_metrics)

In [ ]:
dt_model = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
dt_metrics = train_evaluate_save(dt_model, "Decision Tree",
                                X_train_balanced, y_train_balanced,
                                X_test_scaled, y_test)
performance_results.append(dt_metrics)

## Compare results and pick the best model

In [ ]:
results_df = pd.DataFrame(performance_results)
results_df = results_df.sort_values('Accuracy', ascending=False)

print("\nFINAL MODEL PERFORMANCE:")
print("-" * 60)
print(results_df.to_string(index=False))
print("-" * 60)

best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
print(f"\nBEST MODEL: {best_model_name}")
print(f"Accuracy: {results_df.iloc[0]['Accuracy']:.3f}")
print(f"F1-Score: {results_df.iloc[0]['F1_Score']:.3f}")

## Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1_Score']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx//2, idx%2]
    bars = ax.bar(results_df['Model'], results_df[metric], color=colors)
    ax.set_title(f'{metric}', fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('model_performance_comparison.png', dpi=100, bbox_inches='tight')
print("Saved: model_performance_comparison.png")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices - Error Analysis', fontsize=16, fontweight='bold')

for idx, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred, labels=['PAIDOFF', 'COLLECTION'])

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Predicted PAIDOFF', 'Predicted COLLECTION'],
                yticklabels=['Actual PAIDOFF', 'Actual COLLECTION'],
                ax=axes[idx], annot_kws={"size": 14})

    acc = accuracy_score(y_test, y_pred)
    axes[idx].set_title(f'{name}\nAccuracy: {acc:.3f}', fontweight='bold')
    axes[idx].set_xlabel('Predicted Label', fontweight='bold')
    axes[idx].set_ylabel('True Label', fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=100, bbox_inches='tight')
print("Saved: confusion_matrices.png")

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(dt_model,
          filled=True,
          feature_names=X.columns,
          class_names=['COLLECTION', 'PAIDOFF'],
          rounded=True,
          proportion=True,
          fontsize=10,
          max_depth=3)

plt.title("Decision Tree Structure (First 3 Levels)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('decision_tree_structure.png', dpi=100, bbox_inches='tight')
print("Saved: decision_tree_structure.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if hasattr(lr_model, 'coef_'):
    lr_coef = pd.DataFrame({
        'Feature': X.columns,
        'Coefficient': lr_model.coef_[0]
    }).sort_values('Coefficient', key=abs, ascending=False).head(10)

    colors_lr = ['green' if x > 0 else 'red' for x in lr_coef['Coefficient']]
    axes[0].barh(lr_coef['Feature'], lr_coef['Coefficient'], color=colors_lr)
    axes[0].set_title('Logistic Regression: Feature Impact', fontweight='bold')
    axes[0].set_xlabel('Coefficient Value')
    axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

dt_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

axes[1].barh(dt_importance['Feature'], dt_importance['Importance'], color='purple')
axes[1].set_title('Decision Tree: Feature Importance', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
print("Saved: feature_importance.png")
plt.show()

## Simulated retraining pipeline

This simulates what a periodic retraining job would do: generate new
labeled samples, rebalance, and refit every model. It runs the cycles
inline here rather than on an actual schedule -- wiring this into `cron`
or the `schedule` package for real periodic execution is future work,
not implemented in this notebook.

`generate_augmented_data` draws a random subset of existing rows (with
replacement) and perturbs each with Gaussian noise, so every call
produces a genuinely different synthetic batch instead of always
reusing the same leading rows of the dataset.

In [ ]:
class ModelRetrainer:
    def __init__(self, models_dict, scaler_path='loan_scaler.pkl'):
        self.models = models_dict
        self.scaler = joblib.load(scaler_path)
        self.performance_history = []

    def generate_augmented_data(self, X_data, y_data, n_new_samples=50):
        rng = np.random.default_rng()
        idx = rng.integers(0, len(X_data), size=n_new_samples)
        X_new = np.asarray(X_data)[idx]
        y_new = np.asarray(y_data)[idx]
        noise = rng.normal(0, 0.1, X_new.shape)
        return X_new + noise, y_new

    def retrain_all_models(self, X_train, y_train, X_test, y_test, cycle_num=1):
        print(f"\nRETRAINING CYCLE {cycle_num} - {datetime.now().strftime('%Y-%m-%d %H:%M')}")

        X_aug, y_aug = self.generate_augmented_data(X_train, y_train, n_new_samples=30)
        X_combined = np.vstack([X_train, X_aug])
        y_combined = np.concatenate([y_train, y_aug])

        print(f"Added {len(X_aug)} synthetic samples")
        print(f"New training size: {len(X_combined)} samples")

        X_resampled, y_resampled = SMOTE(random_state=42).fit_resample(X_combined, y_combined)

        cycle_results = []

        for model_name, model in self.models.items():
            model.fit(X_resampled, y_resampled)
            y_pred = model.predict(X_test)
            y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

            metrics = {
                'cycle': cycle_num,
                'timestamp': datetime.now(),
                'model': model_name,
                'accuracy': accuracy_score(y_test, y_pred),
                'f1_score': f1_score(y_test, y_pred, average='weighted'),
                'training_samples': len(X_resampled)
            }

            cycle_results.append(metrics)
            retrained_filename = f"{model_name.lower().replace(' ', '_')}_retrained_cycle{cycle_num}.pkl"
            joblib.dump(model, retrained_filename)

            print(f"{model_name:20} Accuracy: {metrics['accuracy']:.3f} | F1: {metrics['f1_score']:.3f}")

        self.performance_history.extend(cycle_results)
        history_df = pd.DataFrame(self.performance_history)
        history_df.to_csv('retraining_history.csv', index=False)

        print(f"Cycle {cycle_num} complete! Models saved.")

        return cycle_results

    def plot_retraining_history(self):
        if len(self.performance_history) > 1:
            history_df = pd.DataFrame(self.performance_history)

            plt.figure(figsize=(12, 6))

            for model in history_df['model'].unique():
                model_data = history_df[history_df['model'] == model]
                plt.plot(model_data['cycle'], model_data['accuracy'],
                        marker='o', linewidth=2, markersize=8, label=model)

            plt.xlabel('Retraining Cycle', fontweight='bold')
            plt.ylabel('Accuracy', fontweight='bold')
            plt.title('Model Performance Over Retraining Cycles', fontsize=14, fontweight='bold')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.ylim(0.7, 1.0)

            plt.tight_layout()
            plt.savefig('retraining_progress.png', dpi=100, bbox_inches='tight')
            plt.show()
            print("Saved: retraining_progress.png")

In [ ]:
retrainer = ModelRetrainer(models)
initial_retrain = retrainer.retrain_all_models(X_train_scaled, y_train,
                                               X_test_scaled, y_test, cycle_num=1)

for cycle in range(2, 4):
    retrainer.retrain_all_models(X_train_scaled, y_train,
                                 X_test_scaled, y_test, cycle_num=cycle)
    time.sleep(0.5)

retrainer.plot_retraining_history()

## Output summary

In [ ]:
print("\nFILES GENERATED:")
print("Models:")
print("- knn_model.pkl")
print("- logistic_regression_model.pkl")
print("- decision_tree_model.pkl")
print("- *_retrained_cycle*.pkl")
print("\nVisualizations:")
print("- model_performance_comparison.png")
print("- confusion_matrices.png")
print("- decision_tree_structure.png")
print("- feature_importance.png")
print("- retraining_progress.png")
print("\nData Files:")
print("- loan_scaler.pkl")
print("- retraining_history.csv")

print(f"\nBEST MODEL: {best_model_name}")
print(f"Can predict loan defaults with {results_df.iloc[0]['Accuracy']:.1%} accuracy")
print(f"Identifies {results_df.iloc[0]['Recall']:.1%} of actual defaults")